# <center> <img src="../img/ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Procesamiento de Datos Masivos (Big Data)** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Lab 04** </center>
---
**Profesor**: Pablo Camarillo Ramirez

In [9]:
from pcamarillor.spark_utils import SparkUtils
session = SparkUtils("lab 04", "spark://spark-master:7077")
session.spark

In [10]:
# Crete schemas
agencies_schema = SparkUtils.generate_schema([("agency_id", "int"),
                                              ("agency_info", "string")])

brands_schema = SparkUtils.generate_schema([("brand_id", "int"),
                                              ("brand_info", "string")])

car_schema = SparkUtils.generate_schema([("car_id", "int"),
                                              ("car_info", "string")])

customer_schema = SparkUtils.generate_schema([("customer_id", "int"),
                                              ("customer_info", "string")])

rental_schema = SparkUtils.generate_schema([("rental_id", "int"),
                                              ("rental_info", "string")])

In [11]:
# Load Datasets
agencies_df = session.spark.read \
                .schema(agencies_schema) \
                .option("header", "true") \
                .csv("/opt/spark/work-dir/data/car_service/agencies")

brands_df = session.spark.read \
                .schema(brands_schema) \
                .option("header", "true") \
                .csv("/opt/spark/work-dir/data/car_service/brands")

car_df = session.spark.read \
                .schema(car_schema) \
                .option("header", "true") \
                .csv("/opt/spark/work-dir/data/car_service/cars")


customer_df = session.spark.read \
                .schema(customer_schema) \
                .option("header", "true") \
                .csv("/opt/spark/work-dir/data/car_service/customers")

rentals_df = session.spark.read \
                .schema(rental_schema) \
                .option("header", "true") \
                .csv("/opt/spark/work-dir/data/car_service/rentals")

In [12]:
# Extract column from JSON
from pyspark.sql.functions import get_json_object

car_df = car_df.withColumn("car_name", get_json_object(car_df.car_info, "$.car_name"))
agencies_df = agencies_df.withColumn("agency_name", get_json_object(agencies_df.agency_info, "$.agency_name"))
customer_df = customer_df.withColumn("customer_name", get_json_object(customer_df.customer_info, "$.customer_name"))
rentals_df = rentals_df.withColumn("car_id", get_json_object(rentals_df.rental_info, "$.car_id")) \
                       .withColumn("agency_id", get_json_object(rentals_df.rental_info, "$.agency_id")) \
                       .withColumn("customer_id", get_json_object(rentals_df.rental_info, "$.customer_id"))

In [13]:
# Run joins
final_df = rentals_df.join(car_df, on="car_id") \
                        .join(agencies_df, on="agency_id") \
                        .join(customer_df, on="customer_id") \
                        .select("rental_id", "car_name", "agency_name", "customer_name")
final_df.show(5, truncate=False)

+---------+-----------------------+-------------+---------------+
|rental_id|car_name               |agency_name  |customer_name  |
+---------+-----------------------+-------------+---------------+
|11891    |Wallace-Carlson Model 9|NYC Rentals  |Margaret Jones |
|11892    |Grimes-Green Model 8   |LA Car Rental|Albert Williams|
|11893    |Stewart-Allen Model 5  |SF Cars      |Caleb Fleming  |
|11894    |Campos PLC Model 4     |NYC Rentals  |Andrew Butler  |
|11895    |Wagner LLC Model 1     |SF Cars      |Kristin Potts  |
+---------+-----------------------+-------------+---------------+
only showing top 5 rows


In [14]:
final_df.count()

17834

In [15]:
final_df.write \
        .partitionBy("agency_name") \
        .mode("overwrite") \
        .parquet("hdfs://namenode:9000/output/agencies_car_service")

In [16]:
#session.spark.stop()